In [69]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [71]:
import re
import math
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

# ============================================================
# SEED & GPU SETUP
# ============================================================
def set_seed(seed: int = 42):
    """Sets random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (DEVICE.type == 'cuda')

if USE_AMP:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected — running on CPU.")

OPTIONS   = ['A', 'B', 'C', 'D', 'E']
LABEL2IDX = {opt: idx for idx, opt in enumerate(OPTIONS)}
IDX2LABEL = {idx: opt for idx, opt in enumerate(OPTIONS)}

# Load Datasets
# Note: Update these paths if running locally or inside Kaggle
DATA_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/'  # Local path
DATA_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/'       # Kaggle path

train_df = pd.read_csv(f"{DATA_PATH}train.csv")
test_df  = pd.read_csv(f"{DATA_PATH}test.csv")
print(f"Train: {train_df.shape} | Test: {test_df.shape}")

# ============================================================
# TOKENIZER (BPE Trained from Scratch per Fold)
# ============================================================
class BpeTokenizerScratch:
    """
    Fits a Byte-Pair Encoding (BPE) subword tokenizer from scratch on training data.
    This resolves the Out-of-Vocabulary (UNK) issue with technical scientific terms
    without violating 'from-scratch' constraints.
    """
    def __init__(self, max_vocab: int = 20000):
        from tokenizers import Tokenizer as HFTokenizer
        from tokenizers.models import BPE
        from tokenizers.pre_tokenizers import Whitespace
        
        self.tokenizer = HFTokenizer(BPE(unk_token="[UNK]"))
        self.tokenizer.pre_tokenizer = Whitespace()
        self.max_vocab = max_vocab
        self.size = 4  # Initial size based on special tokens

    def build(self, texts: list[str]):
        """Trains the BPE tokenizer from an iterator of raw texts."""
        from tokenizers.trainers import BpeTrainer
        trainer = BpeTrainer(
            vocab_size=self.max_vocab,
            special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]"]
        )
        self.tokenizer.train_from_iterator(texts, trainer)
        self.size = self.tokenizer.get_vocab_size()

    def encode(self, a: str, b: str, max_len: int = 128):
        """
        Tokenizes and pads prompt 'a' and option 'b' into [CLS] + a + [SEP] + b + [SEP].
        Employs smart truncation to preserve the prompt context.
        """
        ta = self.tokenizer.encode(str(a)).ids
        tb = self.tokenizer.encode(str(b)).ids
        
        # [CLS] ta [SEP] tb [SEP] takes 3 special tokens
        total_avail = max_len - 3
        if len(ta) + len(tb) > total_avail:
            # Keep option (tb) up to 48 tokens; allocate remainder to prompt (ta)
            opt_len = min(48, len(tb))
            ta = ta[:total_avail - opt_len]
            tb = tb[:total_avail - len(ta)]
            
        ids = [2] + ta + [3] + tb + [3]
        tids = [0] * (len(ta) + 2) + [1] * (len(tb) + 1)
        pad = max_len - len(ids)
        
        return ids + [0] * pad, tids + [0] * pad, [1] * len(ids) + [0] * pad

def texts_from(df: pd.DataFrame) -> list[str]:
    """Flattens prompts and option texts to fit the tokenizer."""
    return [str(val) for col in ['prompt'] + OPTIONS for val in df[col].tolist()]

# ============================================================
# PRE-COMPUTATION & DATASETS
# ============================================================
def precompute(df: pd.DataFrame, tok: BpeTokenizerScratch, max_len: int, has_labels: bool = True):
    """Encodes the dataset once before training, speeding up the data loader loop."""
    n = len(df)
    ids_arr  = np.zeros((n, 5, max_len), dtype=np.int64)
    tids_arr = np.zeros((n, 5, max_len), dtype=np.int64)
    mask_arr = np.zeros((n, 5, max_len), dtype=np.int64)

    prompts = df['prompt'].astype(str).tolist()
    opt_cols = [df[o].astype(str).tolist() for o in OPTIONS]

    for i in range(n):
        p = prompts[i]
        for j in range(5):
            ids, tids, mask = tok.encode(p, opt_cols[j][i], max_len)
            ids_arr[i, j]  = ids
            tids_arr[i, j] = tids
            mask_arr[i, j] = mask

    labels = None
    if has_labels:
        labels = df['answer'].map(LABEL2IDX).values.astype(np.int64)
        
    return ids_arr, tids_arr, mask_arr, labels


class MCQTensorDataset(Dataset):
    """Loads pre-tokenized numpy arrays directly without overhead."""
    def __init__(self, ids, tids, mask, labels=None):
        self.ids = ids
        self.tids = tids
        self.mask = mask
        self.labels = labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        item = {
            'input_ids'     : torch.from_numpy(self.ids[i]),
            'token_type_ids': torch.from_numpy(self.tids[i]),
            'attention_mask': torch.from_numpy(self.mask[i]),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[i])
        return item

# ============================================================
# MODEL ARCHITECTURE (100% FROM SCRATCH)
# ============================================================
class SinPE(nn.Module):
    """Sinusoidal Positional Encoding."""
    def __init__(self, d: int, max_len: int = 512):
        super().__init__()
        pe  = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MultiHeadAttention(nn.Module):
    """Multi-Head Attention layer with key-padding mask support."""
    def __init__(self, d: int, h: int, drop: float = 0.1):
        super().__init__()
        assert d % h == 0, "d must be divisible by h"
        self.h, self.dh = h, d // h
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.out = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
        
    def forward(self, x, mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        
        split = lambda t: t.view(B, T, self.h, self.dh).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)
        
        s = (q @ k.transpose(-2, -1)) / (self.dh ** 0.5)
        
        if mask is not None:
            # Broadcast mask (B*5, T) -> (B*5, 1, 1, T) to hide padding key tokens
            s = s.masked_fill(mask.unsqueeze(1).unsqueeze(2) == 0, float('-inf'))
            
        a = self.drop(F.softmax(s, dim=-1))
        return self.out((a @ v).transpose(1, 2).reshape(B, T, C))


class TransformerBlock(nn.Module):
    """Standard Pre-LN Transformer encoder block."""
    def __init__(self, d: int, h: int, ff: int, drop: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, h, drop)
        self.ff = nn.Sequential(
            nn.Linear(d, ff),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(ff, d),
            nn.Dropout(drop)
        )
        
    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        return x + self.ff(self.ln2(x))


class ScratchMCQTransformer(nn.Module):
    """
    Scratch MCQ transformer architecture. Runs parallel evaluation
    over all 5 options, returning relative logits for the labels.
    """
    def __init__(self, vocab: int, d: int = 256, h: int = 8, layers: int = 4, ff: int = 512, drop: float = 0.15):
        super().__init__()
        self.emb  = nn.Embedding(vocab, d, padding_idx=0)
        self.temb = nn.Embedding(2, d)
        self.pe   = SinPE(d)
        self.enc  = nn.ModuleList([TransformerBlock(d, h, ff, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(d // 2, 1)
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0, 0.02)
                # CRITICAL: Keep embedding weights of padding token set to 0.0
                if m.padding_idx is not None:
                    with torch.no_grad():
                        m.weight[m.padding_idx].fill_(0.0)

    def encode(self, ids, tids, mask):
        x = self.pe(self.emb(ids) + self.temb(tids))
        for blk in self.enc:
            x = blk(x, mask)
        return self.norm(x)[:, 0]  # Extracts CLS representation

    def forward(self, input_ids, token_type_ids, attention_mask):
        B, N, T = input_ids.shape
        # Flatten option batch dimension to forward concurrently
        cls = self.encode(
            input_ids.reshape(B * N, T),
            token_type_ids.reshape(B * N, T),
            attention_mask.reshape(B * N, T)
        ).view(B, N, -1)
        return self.head(cls).squeeze(-1)

# ============================================================
# EVALUATION METRICS & HELPERS
# ============================================================
def softmax_np(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def map3(y_true, probs):
    scores = []
    for yt, yp in zip(y_true, probs):
        top3 = np.argsort(yp)[::-1][:3]
        s, h = 0., 0
        for i, p in enumerate(top3):
            if p == yt:
                h += 1
                s += h / (i + 1)
        scores.append(s)
    return np.mean(scores)

def warmup_cosine(step, warmup_steps, total_steps):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

# ============================================================
# CONFIGURATION
# ============================================================
CFG = {
    'folds': 5,
    'epochs': 10,
    'bs': 32,
    'lr': 3e-4,             # Increased from 1e-4
    'wd': 0.01,             # Lowered slightly for faster fitting
    'd': 256,               # Increased model capacity
    'h': 8,
    'layers': 4,            # Increased depth
    'ff': 512,              # Increased feedforward dimension
    'max_len': 128,
    'drop': 0.15,           # Lowered dropout to reduce underfitting
    'patience': 3,
    'label_smoothing': 0.1,
    'warmup_frac': 0.1
}

# Cross-Validation Initialization
skf    = StratifiedKFold(CFG['folds'], shuffle=True, random_state=42)
y      = train_df['answer'].map(LABEL2IDX).values
oof    = np.zeros((len(train_df), 5))
test_p = np.zeros((len(test_df), 5))

t_start = time.time()

# ============================================================
# CROSS-VALIDATION LOOP
# ============================================================
for fold, (tri, vli) in enumerate(skf.split(train_df, y)):
    print(f"\n{'='*40}\nFOLD {fold+1}/{CFG['folds']}\n{'='*40}")
    t_fold = time.time()

    # Train a BPE Tokenizer from scratch strictly on current train fold texts
    tok = BpeTokenizerScratch(max_vocab=20000)
    tok.build(texts_from(train_df.iloc[tri]))
    print(f"  Fold BPE vocab size: {tok.size}")

    # Precompute inputs once
    tr_ids, tr_tids, tr_mask, tr_lbl = precompute(train_df.iloc[tri], tok, CFG['max_len'])
    vl_ids, vl_tids, vl_mask, vl_lbl = precompute(train_df.iloc[vli], tok, CFG['max_len'])
    te_ids, te_tids, te_mask, _      = precompute(test_df, tok, CFG['max_len'], has_labels=False)

    tr_ld = DataLoader(MCQTensorDataset(tr_ids, tr_tids, tr_mask, tr_lbl),
                        CFG['bs'], shuffle=True,  num_workers=0, pin_memory=USE_AMP)
    vl_ld = DataLoader(MCQTensorDataset(vl_ids, vl_tids, vl_mask, vl_lbl),
                        CFG['bs'], shuffle=False, num_workers=0, pin_memory=USE_AMP)
    te_ld = DataLoader(MCQTensorDataset(te_ids, te_tids, te_mask),
                        CFG['bs'], shuffle=False, num_workers=0, pin_memory=USE_AMP)

    # Initialize model
    model = ScratchMCQTransformer(
        vocab=tok.size,
        d=CFG['d'],
        h=CFG['h'],
        layers=CFG['layers'],
        ff=CFG['ff'],
        drop=CFG['drop']
    ).to(DEVICE)
    
    # Optional: Re-enable compilation on supported environments
    # try:
    #     model = torch.compile(model)
    # except Exception:
    #     pass

    opt    = torch.optim.AdamW(model.parameters(), CFG['lr'], weight_decay=CFG['wd'])
    scaler = torch.amp.GradScaler(device='cuda', enabled=USE_AMP)
    
    total_steps  = CFG['epochs'] * len(tr_ld)
    warmup_steps = int(CFG['warmup_frac'] * total_steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda s: warmup_cosine(s, warmup_steps, total_steps)
    )

    best, best_state, wait = 0.0, None, 0

    for ep in range(CFG['epochs']):
        # Train Step
        model.train()
        tl, tc, tt = 0.0, 0, 0
        for b in tr_ld:
            ids  = b['input_ids'].to(DEVICE, non_blocking=USE_AMP)
            tids = b['token_type_ids'].to(DEVICE, non_blocking=USE_AMP)
            mask = b['attention_mask'].to(DEVICE, non_blocking=USE_AMP)
            lbl  = b['labels'].to(DEVICE, non_blocking=USE_AMP)
            
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(ids, tids, mask)
                loss = F.cross_entropy(logits, lbl, label_smoothing=CFG['label_smoothing'])
                
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            
            tl += loss.item()
            tc += (logits.argmax(-1) == lbl).sum().item()
            tt += lbl.size(0)

        # Validation Step
        model.eval()
        vlog, vlbl = [], []
        with torch.no_grad():
            for b in vl_ld:
                ids  = b['input_ids'].to(DEVICE, non_blocking=USE_AMP)
                tids = b['token_type_ids'].to(DEVICE, non_blocking=USE_AMP)
                mask = b['attention_mask'].to(DEVICE, non_blocking=USE_AMP)
                lbl  = b['labels'].to(DEVICE, non_blocking=USE_AMP)
                
                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    logits = model(ids, tids, mask)
                vlog.append(logits.cpu().numpy())
                vlbl.append(lbl.cpu().numpy())

        vlog, vlbl = np.vstack(vlog), np.concatenate(vlbl)
        vacc = (vlog.argmax(1) == vlbl).mean()
        vm3  = map3(vlbl, softmax_np(vlog))
        print(f"Ep{ep+1:02d}: loss={tl/len(tr_ld):.3f} acc={tc/tt:.3f} | val_acc={vacc:.3f} map3={vm3:.3f}")

        # Early Stopping check
        if vm3 > best:
            best = vm3
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
            print(f"  ✅ Best MAP@3 Updated: {best:.4f}")
        else:
            wait += 1
            if wait >= CFG['patience']:
                print(f"  Early stopping triggered after {CFG['patience']} epochs without improvement.")
                break

    # Restore best weights and predict OOF + Test
    model.load_state_dict(best_state)
    model.eval()

    vlog = []
    with torch.no_grad():
        for b in vl_ld:
            ids  = b['input_ids'].to(DEVICE, non_blocking=USE_AMP)
            tids = b['token_type_ids'].to(DEVICE, non_blocking=USE_AMP)
            mask = b['attention_mask'].to(DEVICE, non_blocking=USE_AMP)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(ids, tids, mask)
            vlog.append(logits.cpu().numpy())
    oof[vli] = np.vstack(vlog)

    tlog = []
    with torch.no_grad():
        for b in te_ld:
            ids  = b['input_ids'].to(DEVICE, non_blocking=USE_AMP)
            tids = b['token_type_ids'].to(DEVICE, non_blocking=USE_AMP)
            mask = b['attention_mask'].to(DEVICE, non_blocking=USE_AMP)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(ids, tids, mask)
            tlog.append(logits.cpu().numpy())
    test_p += np.vstack(tlog) / CFG['folds']

    print(f"  Fold time: {time.time()-t_fold:.1f}s")
    del model
    if USE_AMP:
        torch.cuda.empty_cache()

# ============================================================
# RESULTS & SUBMISSION
# ============================================================
print(f"\nTotal training time: {time.time()-t_start:.1f}s")

oof_probs = softmax_np(oof)
oof_map3 = map3(y, oof_probs)
print(f"Final Honest OOF MAP@3: {oof_map3:.4f}")

test_probs = softmax_np(test_p)
preds = [' '.join([IDX2LABEL[i] for i in np.argsort(p)[::-1][:3]]) for p in test_probs]
sub = pd.DataFrame({'id': test_df['id'], 'Prediction': preds})
sub.to_csv('submission.csv', index=False)

print("\n--- Submission Sample ---")
print(sub.head())
print("✅ submission.csv saved!")

# Save probabilities for later ensembling (e.g. blending with TF-IDF and DeBERTa)
np.save('scratch_oof_probs.npy', oof_probs)
np.save('scratch_test_probs.npy', test_probs)
print("✅ scratch_oof_probs.npy / scratch_test_probs.npy saved for ensembling.")



✅ GPU detected: Tesla T4
Train: (2000, 8) | Test: (500, 7)

FOLD 1/5



  Fold BPE vocab size: 5691
Ep01: loss=1.579 acc=0.298 | val_acc=0.490 map3=0.619
  ✅ Best MAP@3 Updated: 0.6188
Ep02: loss=1.489 acc=0.362 | val_acc=0.547 map3=0.699
  ✅ Best MAP@3 Updated: 0.6992
Ep03: loss=1.319 acc=0.479 | val_acc=0.767 map3=0.861
  ✅ Best MAP@3 Updated: 0.8608
Ep04: loss=0.878 acc=0.777 | val_acc=0.890 map3=0.934
  ✅ Best MAP@3 Updated: 0.9342
Ep05: loss=0.714 acc=0.865 | val_acc=0.927 map3=0.956
  ✅ Best MAP@3 Updated: 0.9563
Ep06: loss=0.640 acc=0.902 | val_acc=0.930 map3=0.958
  ✅ Best MAP@3 Updated: 0.9575
Ep07: loss=0.599 acc=0.926 | val_acc=0.938 map3=0.961
  ✅ Best MAP@3 Updated: 0.9613
Ep08: loss=0.565 acc=0.935 | val_acc=0.940 map3=0.963
  ✅ Best MAP@3 Updated: 0.9625
Ep09: loss=0.550 acc=0.940 | val_acc=0.938 map3=0.961
Ep10: loss=0.545 acc=0.946 | val_acc=0.938 map3=0.961
  Fold time: 46.6s

FOLD 2/5



  Fold BPE vocab size: 5699
Ep01: loss=1.616 acc=0.276 | val_acc=0.477 map3=0.61